# 01. CICIoT2023 Dataset Setup and Exploratory Analysis
**ML-Powered Intrusion Detection System (IDS) for Secure Network Monitoring**

This notebook inspects and analyzes the actual raw CICIoT2023 dataset located in `data/raw/` across the train, validation, and test partitions.

## 1. Environment & Path Setup

In [ ]:
import sys
import os
import json
import csv
from pathlib import Path
from collections import Counter
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

ROOT_DIR = Path("..").resolve()
RAW_DIR = ROOT_DIR / "data" / "raw"
ANALYSIS_DIR = ROOT_DIR / "results" / "dataset_analysis"
GRAPH_DIR = ROOT_DIR / "results" / "graphs" / "dataset_analysis"

## 2. File Inventory & Storage Footprint

In [ ]:
raw_files = list(RAW_DIR.rglob("*.csv"))
file_info = []
for f in sorted(raw_files):
    size_mb = f.stat().st_size / (1024 * 1024)
    size_gb = size_mb / 1024
    file_info.append({
        "Partition": f.parent.name,
        "Filename": f.name,
        "Path": str(f.relative_to(ROOT_DIR)),
        "Size (MB)": round(size_mb, 2),
        "Size (GB)": round(size_gb, 3)
    })

df_files = pd.DataFrame(file_info)
display(df_files)

## 3. Dataset Overview & Dimensions

In [ ]:
overview_file = ANALYSIS_DIR / "dataset_overview.json"
if overview_file.exists():
    with open(overview_file, "r") as f:
        overview = json.load(f)
    print("Dataset Summary:")
    print(f"  Total Rows:          {overview['total_dataset_rows']:,}")
    print(f"  Total Features:      {overview['num_features']}")
    print(f"  Label Column:        {overview['label_column']}")
    print(f"  Total Classes:       {overview['num_classes']}")
    print(f"  Data Representation: {overview['data_representation']}")
    print(f"  Raw Payload Data:    {overview['is_raw_packet_payload']}")

## 4. Column Schema & Inferred Data Types

In [ ]:
col_summary_file = ANALYSIS_DIR / "column_summary.csv"
if col_summary_file.exists():
    df_cols = pd.read_csv(col_summary_file)
    display(df_cols)

## 5. Sample Records Inspection

In [ ]:
train_path = RAW_DIR / "train" / "train.csv"
if train_path.exists():
    df_sample = pd.read_csv(train_path, nrows=5)
    display(df_sample)

## 6. Complete 34-Class Distribution Across Splits

In [ ]:
total_dist_file = ANALYSIS_DIR / "class_distribution_total.csv"
if total_dist_file.exists():
    df_dist = pd.read_csv(total_dist_file)
    display(df_dist)

## 7. Visualizations: Class Distributions & Partitions

In [ ]:
# Display Top 15 Classes
if (GRAPH_DIR / "class_distribution_top15.png").exists():
    from IPython.display import Image, display
    display(Image(filename=str(GRAPH_DIR / "class_distribution_top15.png")))

# Display Macro Categories
if (GRAPH_DIR / "attack_categories.png").exists():
    display(Image(filename=str(GRAPH_DIR / "attack_categories.png")))

## 8. Data Leakage & Feature Integrity Verification
- **No static identifiers:** No flow/packet ID numbers.
- **No absolute timestamps:** No datetime leakage.
- **No IP/MAC addresses:** Prevents overfitting to specific lab network topology.
- **All 46 features:** Pure statistical flow, protocol, and packet metrics.